In [ ]:
# Imports
from collections import Counter
from collections import defaultdict
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl.utils.dataframe import dataframe_to_rows
from operator import sub
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import types
from sklearn.preprocessing import StandardScaler
import contextlib
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
import gc
import h5py
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import time
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import openpyxl
import os
import pandas as pd
import pickle
import random
import re
import seaborn as sns
import string
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import shutil
import xlsxwriter
from scipy.sparse import csr_matrix
from captum.attr import IntegratedGradients, LayerConductance, LayerActivation
from captum.attr import visualization as viz

from config import get_model_config, COMPLETION_LABELS
from data.load_data import mount_drive, load_data
from data.process_data import filter_proceedings, remove_multi_defendant_cases, create_combined_dockets, filter_to_magistrate_started_cases
from data.sequence_builder import create_enhanced_training_sequences_cached, get_proceeding_sequence_for_defendant
from regex_labels.label_generator_and_cache_manager import generate_and_cache_regex_labels
from models.transformer_creation import create_model_from_data
from training.trainer import train_case_transformer_with_mlb
from training.label_processor import EnhancedLabelProcessor
from training.evaluation import evaluate_with_teacher_forcing
from analysis.cooccurrence import export_cooccurrence_from_valid_completed_cases, create_cooccurrence_matrix_excel
from analysis.frequency_analysis import analyze_raw_label_frequencies_combined, analyze_raw_label_frequencies_singular, print_label_distribution_by_court
from analysis.diagnostic import comprehensive_training_diagnostic
from models.baseline_model import run_baseline
from utils.cache_directory_utils import CacheDirectoryManager
from utils.model_memory_utils import analyze_model_memory, debug_oom_sample
from models.embeddings import extract_and_cache_bert_embeddings_streaming
from analysis.byclerk import analyze_clerk_data

# Set device globally
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Filter to only use completed cases while doing experiments
def filter_to_completed_cases(data, cases, cached_labels):
    completed_cases = []

    # Step 1: Find all proceeding IDs with completion labels
    completed_proceeding_ids = {
        proc_id for proc_id, labels in cached_labels.items()
        if COMPLETION_LABELS.intersection(set(labels))
    }

    # Step 2: Build mag_num -> proceeding_ids mapping
    mag_proceedings = {}
    for _, row in data['mag'].iterrows():
        mag_num = row['mag_num']
        proc_id = row['proceeding_id']
        if mag_num not in mag_proceedings:
            mag_proceedings[mag_num] = []
        mag_proceedings[mag_num].append(proc_id)

    # Step 3: Build case_id -> proceeding_ids mapping
    cdc_proceedings = {}
    for _, row in data['cdc'].iterrows():
        case_id = row['case_id']
        proc_id = row['proceeding_id']
        if case_id not in cdc_proceedings:
            cdc_proceedings[case_id] = []
        cdc_proceedings[case_id].append(proc_id)

    # Step 4: Check each case for completion
    completed_cases = []
    for mag_num, case_id, def_id in cases:
        has_completion = False

        # Check mag proceedings
        if pd.notna(mag_num) and mag_num in mag_proceedings:
            if any(proc_id in completed_proceeding_ids
                   for proc_id in mag_proceedings[mag_num]):
                has_completion = True

        # Check cdc proceedings
        if not has_completion and pd.notna(case_id) and case_id in cdc_proceedings:
            if any(proc_id in completed_proceeding_ids
                   for proc_id in cdc_proceedings[case_id]):
                has_completion = True

        if has_completion:
            completed_cases.append((mag_num, case_id, def_id))

    return completed_cases

## Converts label strings into binary vector representation using label-to-index mapping
def build_case_counts_df(court_data, cached_outcome_labels_v2):
    # 1) which proceedings have labels?
    labeled_pids = {pid for pid, labs in cached_outcome_labels_v2.items() if labs}

    # 2) light views
    mag = court_data['mag'][['mag_num','proceeding_id']].copy()
    cdc = court_data['cdc'][['case_id','proceeding_id']].copy()

    # 3) add is_labeled (vectorized)
    mag['is_labeled'] = mag['proceeding_id'].isin(labeled_pids).values
    cdc['is_labeled'] = cdc['proceeding_id'].isin(labeled_pids).values

    # 4) group once
    mag_counts = (mag.groupby('mag_num', dropna=True)
                    .agg(total_mag=('proceeding_id','size'),
                         labeled_mag=('is_labeled','sum'))
                    .reset_index())

    cdc_counts = (cdc.groupby('case_id', dropna=True)
                    .agg(total_cdc=('proceeding_id','size'),
                         labeled_cdc=('is_labeled','sum'))
                    .reset_index())

    # 5) start from combined_dockets distinct rows
    base = (court_data['combined_dockets']
              [['mag_num','case_id','defendant_id']]
              .drop_duplicates()
              .copy())

    # 6) merge counts; NaNs -> 0
    out = (base.merge(mag_counts, on='mag_num', how='left')
               .merge(cdc_counts, on='case_id',  how='left'))
    for col in ['total_mag','labeled_mag','total_cdc','labeled_cdc']:
        if col in out:
            out[col] = out[col].fillna(0).astype('int64')
        else:
            out[col] = 0

    # 7) totals + validity
    out['total_procs']   = out['total_mag']   + out['total_cdc']
    out['total_labeled'] = out['labeled_mag'] + out['labeled_cdc']
    out['valid'] = (out['total_procs'] >= 2) & (out['total_labeled'] >= 2)

    return out[['mag_num','case_id','defendant_id','total_procs','total_labeled','valid']]

def clean_the_dictionary(dictionary):
    elements_to_remove = model_configuration.EXCLUDED_LABELS
    cleaned_dict = {}
    
    for key, value in dictionary.items():
        if isinstance(value, list):
            # Remove unwanted elements while preserving order
            cleaned_list = [item for item in value if item not in elements_to_remove]
            # Only keep the key if the list is not empty
            if cleaned_list:
                cleaned_dict[key] = cleaned_list
        else:
            # Keep non-list values as they are
            cleaned_dict[key] = value
    
    return cleaned_dict

# ============================================================================
# MAIN EXECUTION STARTS HERE
# ============================================================================

print("\n" + "=" * 80)
print("LOADING AND PROCESSING DATA")
print("=" * 80)

# 1. Mount and load data
print("\n1. Loading data...")
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)
model_configuration = get_model_config()

print("\n2. Filtering proceedings...")
court_data = filter_proceedings(court_data)

print("\n3. Removing multi-defendant cases...")
crawled_original_data_folder_path = Path("/data/krosen5/gdrive-all/tulane-courts-shared/crawled")
court_data, multi_defendant_cases = remove_multi_defendant_cases(court_data, crawled_original_data_folder_path)

print("\n4. Creating combined dockets...")
court_data['combined_dockets'] = create_combined_dockets(court_data)

print("\n5. Filtering to magistrate-started cases...")
court_data = filter_to_magistrate_started_cases(court_data)

print("\n6. Processing regex labels...")
labels_cache = get_cache_dir('labels')
cached_outcome_labels = generate_and_cache_regex_labels(court_data, labels_cache, "load_in")
cached_outcome_labels_v2 = clean_the_dictionary(cached_outcome_labels)    
print(f"   Original labels count: {len(cached_outcome_labels)}")
print(f"   Cleaned labels count: {len(cached_outcome_labels_v2)}")

print("\n7. Building list of defendant cases...")
combined_dockets_df = court_data['combined_dockets']
defendant_case_identifiers = []
for _, row in combined_dockets_df.iterrows():
    case_identifier = (row.get('mag_num'), row.get('case_id'), row.get('defendant_id'))
    defendant_case_identifiers.append(case_identifier)
valid_defendant_case_ids = [
    case_id for case_id in defendant_case_identifiers 
    if pd.notna(case_id[2]) and (pd.notna(case_id[0]) or pd.notna(case_id[1]))
]

print("\n8. Filtering to completed cases...")
completed_defendant_cases = filter_to_completed_cases(court_data, valid_defendant_case_ids, cached_outcome_labels_v2)
print(f"   Found {len(completed_defendant_cases)} completed cases")

print("\n9. Pre-filtering valid cases...")
_counts = build_case_counts_df(court_data, cached_outcome_labels_v2)
_df_cases = pd.DataFrame(completed_defendant_cases, columns=['mag_num','case_id','defendant_id'])
_tagged = (_df_cases.merge(_counts, on=['mag_num','case_id','defendant_id'], how='left')
                    .assign(valid=lambda d: d['valid'].fillna(False)))
valid_completed_defendant_cases = [tuple(x) for x in
    _tagged.loc[_tagged['valid'], ['mag_num','case_id','defendant_id']].to_numpy()]
print(f"   Valid completed cases: {len(valid_completed_defendant_cases)}")


sample_size = min(200, len(valid_completed_defendant_cases))
cases_sample = random.sample(valid_completed_defendant_cases, sample_size)
print(f"\n1. Sampled {sample_size} cases for export")

# Extract proceeding data with labels
print("\n2. Extracting proceedings and labels...")
output_data = []

for mag_num, case_id, def_id in tqdm(cases_sample, desc="Processing cases"):
    # Get all proceedings for this defendant/case
    case_sequences = get_proceeding_sequence_for_defendant(
        def_id, mag_num, case_id, court_data
    )
    
    # Extract proceedings and their labels
    for case in case_sequences:
        for proc in case['proceedings']:
            proc_id = proc.get('proceeding_id')
            labels = cached_outcome_labels_v2.get(proc_id, [])
            
            # Get charge information
            charges_dict = proc.get('charges', {})
            charge_names = [charge_info.get('name', '') for charge_info in charges_dict.values()]
            charge_names_str = ' | '.join(charge_names) if charge_names else ''
            
            output_data.append({
                'mag_num': mag_num if pd.notna(mag_num) else '',
                'case_id': case_id if pd.notna(case_id) else '',
                'defendant_id': def_id,
                'proceeding_id': proc_id,
                'date': proc.get('date'),
                'date_month_year': proc.get('date_month_year', ''),
                'proceeding_type': proc.get('type'),
                'text': proc.get('text', ''),  # First 1000 chars
                'docket_id': proc.get('docket_id', ''),
                'original_bond': proc.get('original_bond', ''),
                'sum_of_bonds': proc.get('sum_of_bonds', ''),
                'race_id': proc.get('race_id', ''),
                'gender_id': proc.get('gender_id', ''),
                'age': proc.get('age', ''),
                'magistrate_section': proc.get('magistrate_section', ''),
                'charges': charge_names_str,
                'num_charges': len(charges_dict),
                'labels': '|'.join(labels) if labels else 'NO_LABELS',
                'label_count': len(labels)
            })

# Create DataFrame
print("\n3. Creating DataFrame...")
output_df = pd.DataFrame(output_data)

# Sort by defendant and date
output_df = output_df.sort_values(['defendant_id', 'date']).reset_index(drop=True)

# Save to CSV
output_path = get_cache_dir('exports') / 'case_labels_sample_200_completed.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_df.to_csv(output_path, index=False)

print("\n" + "=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)
print(f"✓ File saved to: {output_path}")
print(f"\nStatistics:")
print(f"  - Total cases exported: {len(cases_sample)}")
print(f"  - Total proceedings: {len(output_df)}")
print(f"  - Proceedings with labels: {len(output_df[output_df['label_count'] > 0])}")
print(f"  - Average labels per proceeding: {output_df['label_count'].mean():.2f}")
print(f"  - Unique defendants: {output_df['defendant_id'].nunique()}")
print(f"  - Date range: {output_df['date'].min()} to {output_df['date'].max()}")

# Print label distribution
print(f"\nTop 10 Most Common Labels:")
all_labels = []
for labels_str in output_df[output_df['labels'] != 'NO_LABELS']['labels']:
    all_labels.extend(labels_str.split('|'))

label_counts = Counter(all_labels)
for label, count in label_counts.most_common(10):
    print(f"  {label}: {count}")

# Print sample of the data
print(f"\nFirst 5 rows preview:")
print(output_df[['defendant_id', 'proceeding_id', 'date', 'proceeding_type', 'label_count', 'labels']].head())

print("\n" + "=" * 80)
print("DONE!")
print("=" * 80)

Using device: cuda
GPU: NVIDIA RTX A5000
CUDA Version: 12.4

LOADING AND PROCESSING DATA

1. Loading data...
✓ Found Google Drive mount at /data/krosen5/gdrive-all
✓ Google Drive mount is accessible
✓ Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2

2. Filtering proceedings...

3. Removing multi-defendant cases...
Removed 2048 CDC and 1881 mag cases (including corresponding matches)

4. Creating combined dockets...

5. Filtering to magistrate-started cases...
Starting with 127421 total cases in combined_dockets
Found 122635 magistrate-started cases
  Filtered mag proceedings: 153004 -> 152805
  Filtered mag_dockets: 122793 -> 122635
  Filtered mag_charges: 284218 -> 283964
  Filtered cdc proceedings: 165729 -> 158057
  Filtered cdc_dockets: 25184 -> 20480
  Filtered cdc_charges: 60446 -> 49447
  Filtered defendants: 74934 -> 70579
Filtering complete. Dataset now contains only magistrate-started cases.

6. Processing regex labels...
Loading labels cache from /data

100%|██████████| 158057/158057 [00:03<00:00, 41777.28it/s]


   Original labels count: 310862
   Cleaned labels count: 173719

7. Building list of defendant cases...

8. Filtering to completed cases...
   Found 24647 completed cases

9. Pre-filtering valid cases...
   Valid completed cases: 24323

1. Sampled 200 cases for export

2. Extracting proceedings and labels...


Processing cases: 100%|██████████| 200/200 [00:40<00:00,  4.89it/s]



3. Creating DataFrame...

EXPORT COMPLETE
✓ File saved to: /data/krosen5/cache/transformers/Transformer/exports/case_labels_sample_200_completed.csv

Statistics:
  - Total cases exported: 200
  - Total proceedings: 1572
  - Proceedings with labels: 955
  - Average labels per proceeding: 1.08
  - Unique defendants: 200
  - Date range: 2014-03-01 00:00:00 to 2025-09-18 00:00:00

Top 10 Most Common Labels:
  BOND_SET: 233
  CONTINUANCE_OCCURRED: 233
  CHARGES_REFUSED: 218
  FIRST_APPEARANCE_HEARING_OCCURRED: 200
  THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER: 188
  PRE_TRIAL_CONFERENCE_HELD: 155
  PROBABLE_CAUSE_FOUND: 77
  DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE: 57
  PLED_NOT_GUILTY_TO_CHARGES: 55
  DOMESTIC_STAY_AWAY_ORDER_IMPOSED: 49

First 5 rows preview:
  defendant_id proceeding_id       date proceeding_type  label_count  \
0        27521    cdc_156664 2020-07-28             cdc            1   
1        27521    cdc_156665 2020-08-05             cdc            3   
2        2752